# Taylor Swift Eras: Lyrical Themes Alluvial Diagram

Instead of visualizing the raw flow of individual emotions (e.g. Joy, Sadness), this diagram groups the recurring **emotion combinations** into distinct **Lyrical Themes** to answer:
> *What broader lyrical themes can be inferred from recurring emotion combinations, and how do these themes vary across albums?*

### Methodology
1. **Theme Discovery (K-Means)**: We run a K-Means clustering algorithm ($k=6$) on the emotional fingerprints of all individual sections. This uncovers 6 naturally occurring thematic clusters in her discography (e.g., when Sadness and Anger spike together, the algorithm identifies the "Heartbreak & Betrayal" theme).
2. **Theme Assignment**: Every verse, chorus, and bridge is labeled with one of these 6 Lyrical Themes.
3. **Optimal Transport Flow**: We use the exact same Optimal Transport (Earth Mover's Distance) algorithm from before to map the emotional mass of Album A to Album B. But this time, we aggregate the flow through the lens of the **Themes**. This shows us, for example, exactly how the "Nostalgia" sections in *Folklore* evolved into the "Anxiety" sections in *Midnights*.

In [ ]:
import json
from pathlib import Path
import numpy as np
import plotly.graph_objects as go

In [ ]:
data_dir = Path("data")
with open(data_dir / "section_emotion_wheel_data.llm.json", "r", encoding="utf-8") as f:
    sections = json.load(f)
    
EMOTION_KEYS = ["joy", "trust", "fear", "surprise", "sadness", "disgust", "anger", "anticipation"]
selected_albums = ["Fearless", "Red", "1989", "Reputation", "Folklore", "Midnights"]

# Extract 8D vectors for all sections
X = np.array([[s['emotion_scores'].get(e, 0.0) for e in EMOTION_KEYS] for s in sections])

## 1. Discovering Themes (K-Means Clustering)
We implement a standard K-Means algorithm to find the 6 dominant emotional clusters.

In [ ]:
def kmeans(X, k, max_iters=100):
    np.random.seed(42)
    centroids = X[np.random.choice(X.shape[0], k, replace=False)]
    for _ in range(max_iters):
        distances = np.linalg.norm(X[:, np.newaxis] - centroids, axis=2)
        labels = np.argmin(distances, axis=1)
        new_centroids = np.array([X[labels == i].mean(axis=0) if np.sum(labels == i) > 0 else centroids[i] for i in range(k)])
        if np.all(centroids == new_centroids):
            break
        centroids = new_centroids
    return centroids, labels

centroids, cluster_labels = kmeans(X, k=6)

## 2. Naming the Themes
Based on the top 2 emotions of each centroid, we can infer a descriptive lyrical theme.

In [ ]:
def get_theme_info(centroid, emotion_keys):
    top_indices = np.argsort(centroid)[::-1][:2]
    top_emotions = set([emotion_keys[idx] for idx in top_indices])
    
    if "sadness" in top_emotions and "fear" in top_emotions:
        return "Anxiety & Vulnerability", "#708090" # Slate Grey
    if "joy" in top_emotions and "trust" in top_emotions:
        return "Romantic Love & Devotion", "#FFB6C1" # Light Pink
    if "anger" in top_emotions and "sadness" in top_emotions:
        return "Heartbreak & Betrayal", "#8B0000" # Dark Red
    if "joy" in top_emotions and "anticipation" in top_emotions:
        return "Hope & Infatuation", "#FFD700" # Gold
    if "anticipation" in top_emotions and "fear" in top_emotions:
        return "Suspense & Uncertainty", "#20B2AA" # Light Sea Green
    if "sadness" in top_emotions and "anticipation" in top_emotions:
        return "Nostalgia & Longing", "#4682B4" # Steel Blue
        
    return "Other", "#D3D3D3"

# Assign a Theme Name and Color to each of the 6 clusters
cluster_themes = []
cluster_colors = []
for i, center in enumerate(centroids):
    name, color = get_theme_info(center, EMOTION_KEYS)
    cluster_themes.append(name)
    cluster_colors.append(color)

for i, name in enumerate(cluster_themes):
    print(f"Cluster {i+1}: {name}")
    
# Attach the assigned theme back to the sections
for i, section in enumerate(sections):
    section['theme'] = cluster_themes[cluster_labels[i]]
    section['theme_color'] = cluster_colors[cluster_labels[i]]

## 3. Optimal Transport Flow
We calculate the Earth Mover's Distance between sections in consecutive albums, and aggregate the micro-flows by their assigned Theme.

In [ ]:
def sinkhorn_knopp(M, reg=0.05, numItermax=1000, stopThr=1e-9):
    n_a, n_b = M.shape
    a = np.ones(n_a) / n_a
    b = np.ones(n_b) / n_b
    K = np.exp(-M / reg)
    u = np.ones(n_a) / n_a
    
    for i in range(numItermax):
        u_prev = u
        v = b / (np.dot(K.T, u))
        u = a / (np.dot(K, v))
        if np.max(np.abs(u - u_prev)) < stopThr:
            break
    return np.diag(u) @ K @ np.diag(v)

node_labels = []
node_colors = []
node_map = {}
node_idx = 0

# Create nodes for all Themes across the 6 selected eras
unique_themes = list(set(cluster_themes))

for stage_idx, album in enumerate(selected_albums):
    for theme in unique_themes:
        # Only add a node if the theme exists in this album to prevent empty nodes
        if any(s['theme'] == theme for s in sections if s['album_title'] == album):
            node_labels.append(f"{theme} ({album})")
            color = next(s['theme_color'] for s in sections if s['theme'] == theme)
            node_colors.append(color)
            node_map[(stage_idx, theme)] = node_idx
            node_idx += 1

sources = []
targets = []
values = []
link_colors = []

for stage_idx in range(len(selected_albums) - 1):
    album_A = selected_albums[stage_idx]
    album_B = selected_albums[stage_idx + 1]
    
    secs_A = [s for s in sections if s['album_title'] == album_A]
    secs_B = [s for s in sections if s['album_title'] == album_B]
    
    X_A = np.array([[s['emotion_scores'].get(e, 0.0) for e in EMOTION_KEYS] for s in secs_A])
    X_B = np.array([[s['emotion_scores'].get(e, 0.0) for e in EMOTION_KEYS] for s in secs_B])
    
    M = np.linalg.norm(X_A[:, np.newaxis, :] - X_B[np.newaxis, :, :], axis=2)
    M = M / np.max(M) 
    
    P = sinkhorn_knopp(M, reg=0.05)
    
    flow_agg = {}
    for i, s_A in enumerate(secs_A):
        for j, s_B in enumerate(secs_B):
            t_A = s_A['theme']
            t_B = s_B['theme']
            key = (t_A, t_B)
            flow_agg[key] = flow_agg.get(key, 0.0) + P[i, j]
                
    for (t_A, t_B), flow_val in flow_agg.items():
        if flow_val < 0.01: # Filter tiny noise flows
            continue
            
        color_hex = next(s['theme_color'] for s in sections if s['theme'] == t_A).lstrip('#')
        r, g, b = int(color_hex[0:2], 16), int(color_hex[2:4], 16), int(color_hex[4:6], 16)
        
        alpha = 0.55 if t_A == t_B else (0.4 if flow_val >= 0.03 else 0.1)
            
        sources.append(node_map[(stage_idx, t_A)])
        targets.append(node_map[(stage_idx + 1, t_B)])
        values.append(flow_val)
        link_colors.append(f"rgba({r}, {g}, {b}, {alpha})")

fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=20,
        thickness=25,
        line=dict(color="black", width=0.5),
        label=node_labels,
        color=node_colors
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values,
        color=link_colors
    )
)])

fig.update_layout(
    title_text="Taylor Swift Eras: Lyrical Themes Evolution (Optimal Transport)",
    font_size=12,
    width=1200,
    height=800,
    plot_bgcolor='white',
    paper_bgcolor='white'
)
fig.show()